# Projeto Final: Otimização de Inventário DataCo
## Fase 1: Limpeza e Preparação de Dados
Nesta etapa, realizamos a ingestão dos datasets de transações (ERP) e logs de comportamento digital (Web). O foco é garantir a integridade dos dados e identificar a estrutura necessária para correlacionar o interesse dos clientes com a venda física.

In [1]:
import pandas as pd

# 1. Carregamento dos dados de vendas e logs com tratamento de encoding
# Usamos 'latin1' para evitar erros em caracteres especiais dos nomes dos produtos
df_vendas = pd.read_csv('../dados/DataCoSupplyChainDataset.csv', encoding='latin1')
df_logs = pd.read_csv('../dados/tokenized_access_logs.csv', encoding='latin1')


print("Ficheiros carregados com sucesso!")


Ficheiros carregados com sucesso!


In [2]:
# 2. Auditoria de Tipos e Nulos
# Precisamos de saber se as colunas de data e ID estão preenchidas
print(f"\n--- AUDITORIA INICIAL ---")
print(f"Vendas: {df_vendas.shape[0]} registos | Colunas com Nulos: {df_vendas.isnull().any().sum()}")
print(f"Logs: {df_logs.shape[0]} registos | Colunas com Nulos: {df_logs.isnull().any().sum()}")




--- AUDITORIA INICIAL ---
Vendas: 180519 registos | Colunas com Nulos: 4
Logs: 469977 registos | Colunas com Nulos: 0


In [3]:
# 3. Verificação do Estado dos Produtos
# Verificamos se os produtos estão ativos (0 = Ativo)
print("\nDistribuição de Status de Produto:")
print(df_vendas['Product Status'].value_counts())




Distribuição de Status de Produto:
Product Status
0    180519
Name: count, dtype: int64


In [4]:
# 4. Amostra para validação visual
print(f"\n--- DATASET DE VENDAS (ERP) | Dimensões: {df_vendas.shape} ---")
display(df_vendas.head())

print(f"\n--- DATASET DE LOGS (WEB) | Dimensões: {df_logs.shape} ---")
display(df_logs.head())


--- DATASET DE VENDAS (ERP) | Dimensões: (180519, 53) ---


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class



--- DATASET DE LOGS (WEB) | Dimensões: (469977, 8) ---


,Product,Category,Date,Month,Hour,Department,ip,url
0,adidas Brazuca 2017 Official Match Ball,baseball & softball,9/1/2017 6:00,Sep,6,fitness,37.97.182.65,/department/fitness/category/baseball%20&%20so...
1,The North Face Women's Recon Backpack,hunting & shooting,9/1/2017 6:00,Sep,6,fan shop,206.56.112.1,/department/fan%20shop/category/hunting%20&%20...
2,adidas Kids' RG III Mid Football Cleat,featured shops,9/1/2017 6:00,Sep,6,apparel,215.143.180.0,/department/apparel/category/featured%20shops/...
3,Under Armour Men's Compression EV SL Slide,electronics,9/1/2017 6:00,Sep,6,footwear,206.56.112.1,/department/footwear/category/electronics/prod...
4,Pelican Sunstream 100 Kayak,water sports,9/1/2017 6:01,Sep,6,fan shop,136.108.56.242,/department/fan%20shop/category/water%20sports...


### 1. Seleção de Variáveis Estratégicas
Para otimizar a performance do modelo, reduzimos o dataset de 53 para 14 variáveis críticas. A seleção baseou-se em três dimensões fundamentais:
* **Dimensão de Procura:** Volume de itens e valor de vendas.
* **Dimensão Logística:** Tempos de envio (real vs. planeado) e modo de transporte.
* **Dimensão de Risco e Rentabilidade:** Indicadores de atraso e margem por encomenda.

In [5]:
# 1. Definição das colunas essenciais
colunas_finais = [
    'order date (DateOrders)', 'Product Name', 'Category Name', 'Order Item Quantity', 
    'Product Status', 'Days for shipping (real)', 'Days for shipment (scheduled)', 
    'Order Status', 'Late_delivery_risk', 'Shipping Mode', 'Benefit per order', 
    'Product Price', 'Order Region', 'Sales'
]


df_vendas = df_vendas[colunas_finais].copy()

# 2. Renomear a coluna de data para facilitar o código futuro
df_vendas = df_vendas.rename(columns={'order date (DateOrders)': 'order_date'})

print(f"✅ Seleção concluída. O dataset agora tem {df_vendas.shape[1]} colunas.")
print(f"Colunas finais: {list(df_vendas.columns)}")

✅ Seleção concluída. O dataset agora tem 14 colunas.
Colunas finais: ['order_date', 'Product Name', 'Category Name', 'Order Item Quantity', 'Product Status', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Order Status', 'Late_delivery_risk', 'Shipping Mode', 'Benefit per order', 'Product Price', 'Order Region', 'Sales']


### 2. Tratamento Temporal
Converti as datas para formato datetime tendo em conta que são utilizados formatos de data diferentes (formato americano)

In [6]:
# 1. Converter Vendas
# Normalizar para remover horas e manter apenas o dia
df_vendas['order_date'] = pd.to_datetime(df_vendas['order_date']).dt.normalize()

# 2. Cálculo do Atraso Logístico (atraso_delta)
df_vendas['atraso_delta'] = df_vendas['Days for shipping (real)'] - df_vendas['Days for shipment (scheduled)']

# 3. Converter Logs (formato americano)
df_logs['order_date'] = pd.to_datetime(df_logs['Date'], format='%m/%d/%Y %H:%M', errors='coerce').dt.normalize()


print(f"Janela Vendas: {df_vendas['order_date'].min().date()} a {df_vendas['order_date'].max().date()}")
print(f"Janela Logs: {df_logs['order_date'].min().date()} a {df_logs['order_date'].max().date()}")

Janela Vendas: 2015-01-01 a 2018-01-31
Janela Logs: 2017-09-01 a 2018-01-31


### 3. Refinação de Dados e Identificação de Produtos Críticos
Nesta etapa, removemos o "ruído" transacional e normalizamos os identificadores para garantir que o sinal digital (cliques) corresponde exatamente à transação física.

Integridade de Correspondência: Normalização de strings para evitar que "Product A" e "product a " sejam lidos como itens diferentes.

Filtro de Procura Efetiva: Focamos em estados de encomenda que representam intenção real de compra, eliminando cancelamentos que distorceriam a previsão de stock.

In [7]:
# 1. Normalização de Identificadores (Strip e Lowercase para segurança total)
df_vendas['Product Name'] = df_vendas['Product Name'].str.strip()
df_logs['Product'] = df_logs['Product'].str.strip()

# 2. RANKING DE VULNERABILIDADE
# Separamos por tipo de problema
ranking_cancelados = df_vendas[df_vendas['Order Status'] == 'CANCELED']['Product Name'].value_counts()
ranking_atrasos = df_vendas[df_vendas['Late_delivery_risk'] == 1]['Product Name'].value_counts()

# Criar um DataFrame de problemas consolidado
ranking_problemas = pd.DataFrame({
    'Cancelamentos': ranking_cancelados,
    'Riscos_Atraso': ranking_atrasos
}).fillna(0)
ranking_problemas['Total_Problemas'] = ranking_problemas['Cancelamentos'] + ranking_problemas['Riscos_Atraso']
ranking_problemas = ranking_problemas.sort_values(by='Total_Problemas', ascending=False).reset_index()
ranking_problemas.rename(columns={'index': 'Product Name'}, inplace=True)

# 3. SANEAMENTO DE PROCURA REAL
# Mantemos apenas o que é venda efetiva ou em processamento
status_validos = ['COMPLETE', 'CLOSED', 'PROCESSING', 'PENDING', 'PENDING_PAYMENT']
df_vendas = df_vendas[df_vendas['Order Status'].isin(status_validos)].copy()

print(f"✅ Normalização concluída.")
print(f"Dataset de Vendas filtrado para Procura Real: {df_vendas.shape[0]} registos.")
print(f"Top 3 Produtos Críticos identificados:\n{ranking_problemas.head(3)}")

✅ Normalização concluída.
Dataset de Vendas filtrado para Procura Real: 161068 registos.
Top 3 Produtos Críticos identificados:
                              Product Name  Cancelamentos  Riscos_Atraso  \
0         Perfect Fitness Perfect Rip Deck          477.0          13473   
1  Nike Men's CJ Elite 2 TD Football Cleat          467.0          12121   
2     Nike Men's Dri-FIT Victory Golf Polo          438.0          11476   

   Total_Problemas  
0          13950.0  
1          12588.0  
2          11914.0  


### 4. Identificação de "Ruturas Virtuais" de Stock
Como o dataset não fornece dados diretos de inventário ou ruturas, desenvolvi uma métrica Proxy para identificar falhas de disponibilidade. Considerei uma "Rutura Virtual" quando se verificam cumulativamente dois critérios:

Late_delivery_risk == 1: O sistema sinalizou um risco real de atraso na entrega.

atraso_delta >= 2: O atraso efetivo no envio foi igual ou superior a 2 dias em relação ao planeado.

Justificação Técnica: Optei por um limiar de 2 dias para isolar ruídos logísticos menores (ex: trânsito ou atrasos do estafeta). Um atraso superior seria um indicador forte de que o produto não estava disponível no momento do pedido.

In [8]:
# 1. Definir a condição de "Rutura Virtual" usando o Risco de Atraso
# Se Late_delivery_risk == 1 E o atraso real (atraso_delta) for >= 2 dias
condicao_rutura = (df_vendas['Late_delivery_risk'] == 1) & (df_vendas['atraso_delta'] >= 2)

# 2. Contar as ocorrências
total_casos = condicao_rutura.sum()
percentagem = (total_casos / len(df_vendas)) * 100

print(f"--- RESULTADO DA ANÁLISE DE RUTURAS (Proxy) ---")
print(f"Total de encomendas com atraso crítico (>= 2 dias): {total_casos}")
print(f"Representa {percentagem:.2f}% do total de vendas.")

# 3. Mostrar os produtos que mais sofrem com isto
if total_casos > 0:
    top_ruturas = df_vendas[condicao_rutura]['Product Name'].value_counts().head(10)
    print("\nProdutos com maior incidência de rutura (Top 10):")
    print(top_ruturas)
else:
    print("\nAVISO: Critério demasiado restrito.")

--- RESULTADO DA ANÁLISE DE RUTURAS (Proxy) ---
Total de encomendas com atraso crítico (>= 2 dias): 38309
Representa 23.78% do total de vendas.

Produtos com maior incidência de rutura (Top 10):
Product Name
Perfect Fitness Perfect Rip Deck                 5310
Nike Men's CJ Elite 2 TD Football Cleat          4768
Nike Men's Dri-FIT Victory Golf Polo             4327
O'Brien Men's Neoprene Life Vest                 4072
Field & Stream Sportsman 16 Gun Fire Safe        3662
Pelican Sunstream 100 Kayak                      3250
Diamondback Women's Serene Classic Comfort Bi    2918
Nike Men's Free 5.0+ Running Shoe                2559
Under Armour Girls' Toddler Spine Surge Runni    2279
Fighting video games                              181
Name: count, dtype: int64


#### 4.1. Seleção Estratégica dos Dados
Para garantir a eficácia do modelo de Machine Learning, selecionei os 5 produtos que apresentam o equilíbrio ideal entre:

- Elevada frequência de ruturas virtuais.

- Volume de logs de cliques para treino.

Este cruzamento de dados (Score de Viabilidade) garante que o modelo tem exemplos suficientes para aprender padrões de comportamento de consumo.

In [9]:
# 1. Contar cliques por produto
contagem_cliques = df_logs['Product'].value_counts().reset_index()
contagem_cliques.columns = ['Product Name', 'Total_Clicks']

# 2. Contar ruturas por produto
df_ruturas_prod = df_vendas[condicao_rutura]['Product Name'].value_counts().reset_index()
df_ruturas_prod.columns = ['Product Name', 'Total_Ruturas']

# 3. Cruzar dados (Apenas produtos que têm vendas E cliques)
check_validacao = pd.merge(df_ruturas_prod, contagem_cliques, on='Product Name', how='inner')

# 4. Criar o Score de Viabilidade
check_validacao['Score_Viabilidade'] = check_validacao['Total_Ruturas'] * check_validacao['Total_Clicks']
ranking_final = check_validacao.sort_values(by='Score_Viabilidade', ascending=False)

# 5. DEFINIÇÃO FINAL DA LISTA
lista_produtos_criticos = ranking_final.head(5)['Product Name'].tolist()

print("--- TOP 5 PRODUTOS SELECIONADOS PARA DEMAND SENSING ---")
print(ranking_final.head(5))
print(f"\nLista final para o modelo: {lista_produtos_criticos}")

--- TOP 5 PRODUTOS SELECIONADOS PARA DEMAND SENSING ---
                                Product Name  Total_Ruturas  Total_Clicks  \
0           Perfect Fitness Perfect Rip Deck           5310         27878   
1    Nike Men's CJ Elite 2 TD Football Cleat           4768         25241   
2       Nike Men's Dri-FIT Victory Golf Polo           4327         25627   
3           O'Brien Men's Neoprene Life Vest           4072         16194   
4  Field & Stream Sportsman 16 Gun Fire Safe           3662         15178   

   Score_Viabilidade  
0          148032180  
1          120349088  
2          110888029  
3           65941968  
4           55581836  

Lista final para o modelo: ['Perfect Fitness Perfect Rip Deck', "Nike Men's CJ Elite 2 TD Football Cleat", "Nike Men's Dri-FIT Victory Golf Polo", "O'Brien Men's Neoprene Life Vest", 'Field & Stream Sportsman 16 Gun Fire Safe']


### 5. Agregação Diária e Foco nos Produtos Críticos
Nesta etapa final de preparação, transpomos a lógica de negócio para uma estrutura de séries temporais, focando o projeto no período onde a correlação entre o comportamento digital e a venda física é máxima.

Escolha Temporal (SETEMBRO/OUTUBRO): Embora o dataset contenha registos até 2018 (Janeiro), a auditoria de qualidade revelou que a sincronização entre os logs de cliques e os registos do ERP só existe nesta janela. Restringir a análise a este período garante que o modelo de Demand Sensing aprenda com dados reais.

Agregação de Vendas: Consolidação da quantidade vendida por dia para os 5 produtos críticos identificados no ranking de vulnerabilidade.

Agregação de Logs: Transformei os registos de acesso individuais em contagens diárias de cliques por produto.

Alinhamento Temporal: Utilizei um left merge para garantir a integridade da série temporal. Isto permite que o modelo aprenda que mesmo em dias com vendas, a ausência de cliques (registada como zero) é um sinal importante para a previsão.

In [10]:
# 1. Agregação de LOGS (Cliques diários por produto)
logs_diarios = df_logs[df_logs['Product'].isin(lista_produtos_criticos)].copy()
logs_diarios = logs_diarios.groupby(['order_date', 'Product'])['ip'].count().reset_index()
logs_diarios.columns = ['order_date', 'Product Name', 'clicks']

# 2. Agregação de VENDAS (Quantidades diárias por produto)
vendas_diarias = df_vendas[df_vendas['Product Name'].isin(lista_produtos_criticos)].copy()
vendas_diarias = vendas_diarias.groupby(['order_date', 'Product Name'])['Order Item Quantity'].sum().reset_index()

# 3. Cruzamento de Dados (Alinhamento Temporal)
# O 'left merge' garante que mantemos os dias de venda mesmo sem cliques (preenchidos com 0)
df_final = pd.merge(vendas_diarias, logs_diarios, on=['order_date', 'Product Name'], how='left')
df_final['clicks'] = df_final['clicks'].fillna(0)

# 4. Aplicação do Filtro da Janela Temporal
# Limitamos ao período de sincronização total (01/09/2017 a 02/10/2017)
df_final = df_final[(df_final['order_date'] >= '2017-09-01') & (df_final['order_date'] <= '2017-10-02')]

# 5. Ordenação Cronológica e Exportação
df_final = df_final.sort_values(by=['Product Name', 'order_date']).reset_index(drop=True)
caminho_dados = '../dados/dataset_final_modelacao.csv'
df_final.to_csv(caminho_dados, index=False)

# 6. Sumário de Validação para o Relatório
print(f"✅ Sincronização Concluída para os 5 Produtos Críticos.")
print(f"Janela de Estudo: {df_final['order_date'].min().date()} a {df_final['order_date'].max().date()}")
print(f"Total de amostras diárias: {len(df_final)}")
display(df_final.head(10))

✅ Sincronização Concluída para os 5 Produtos Críticos.
Janela de Estudo: 2017-09-01 a 2017-10-02
Total de amostras diárias: 160


,order_date,Product Name,Order Item Quantity,clicks
0,2017-09-01,Field & Stream Sportsman 16 Gun Fire Safe,11,95.0
1,2017-09-02,Field & Stream Sportsman 16 Gun Fire Safe,12,106.0
2,2017-09-03,Field & Stream Sportsman 16 Gun Fire Safe,17,74.0
3,2017-09-04,Field & Stream Sportsman 16 Gun Fire Safe,14,76.0
4,2017-09-05,Field & Stream Sportsman 16 Gun Fire Safe,18,84.0
5,2017-09-06,Field & Stream Sportsman 16 Gun Fire Safe,14,82.0
6,2017-09-07,Field & Stream Sportsman 16 Gun Fire Safe,12,83.0
7,2017-09-08,Field & Stream Sportsman 16 Gun Fire Safe,11,104.0
8,2017-09-09,Field & Stream Sportsman 16 Gun Fire Safe,13,75.0
9,2017-09-10,Field & Stream Sportsman 16 Gun Fire Safe,18,81.0


In [11]:
# 1. Garantir que as datas estão no formato correto
df_vendas['order_date'] = pd.to_datetime(df_vendas['order_date'])

# 2. Criar a lógica de falha (Proxy) no dataset
df_vendas['is_falha'] = 0

# Usamos a TUA coluna 'Late_delivery_risk' == 1 em vez do texto
mask_falha = ((df_vendas['Days for shipping (real)'] - df_vendas['Days for shipment (scheduled)']) >= 2) & \
             (df_vendas['Late_delivery_risk'] == 1)

df_vendas.loc[mask_falha, 'is_falha'] = 1

# 3. Agregação Diária para o BI
# ATENÇÃO: Se renomeaste a coluna 'Order Item Quantity' para 'Quantity', 
# muda o nome aqui debaixo na agregação!
dados_bi = df_vendas.groupby(df_vendas['order_date'].dt.date).agg({
    'Order Item Quantity': 'sum', 
    'is_falha': 'sum'
}).reset_index()

dados_bi.columns = ['Data', 'Vendas Totais', 'Falhas Totais']

# 4. Exportar para a pasta da App
dados_bi.to_csv('../app/dados_bi.csv', index=False)
print("✅ Sucesso! O ficheiro dados_bi.csv foi finalmente gerado!")

✅ Sucesso! O ficheiro dados_bi.csv foi finalmente gerado!
